# Synthetic Data Generation (dbldatagen)

Notebook for testing synthetic data generation. Mirrors the logic in `src/synthetic_data.py`.

**Use case:** Generate test data (orders, order_details) for DLT-Meta pipelines without external sources.

## Setup

In [ ]:
# Optional: Define widget for Databricks (skipped if dbutils not available)
try:
    dbutils.widgets.text("output_location", "/tmp/synthetic_data", "output_location")
except NameError:
    pass

In [ ]:
%pip install --quiet dbldatagen

In [ ]:
import dbldatagen as dg
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("SyntheticDataGeneration").getOrCreate()

# Configuration (use dbutils on Databricks, or default for local testing)
try:
    output_location = dbutils.widgets.get("output_location") or "/tmp/synthetic_data"
except NameError:
    output_location = "/tmp/synthetic_data"
output_format = "parquet"
schema_output_location = f"{output_location}/_schemas"

print(f"Output: {output_location}")
print(f"Format: {output_format}")
print(f"Schema: {schema_output_location}")

## Create Output Directories

In [ ]:
try:
    dbutils.fs.mkdirs(output_location)
    dbutils.fs.mkdirs(schema_output_location)
    print("Created output directories (Databricks)")
except NameError:
    import os
    os.makedirs(output_location, exist_ok=True)
    os.makedirs(schema_output_location, exist_ok=True)
    print("Created output directories (local)")

## Generate Orders Table

In [ ]:
spec_orders = dg.DataGenerator(spark, rows=1000, partitions=2)
spec_orders = spec_orders.withColumn("order_id", "long", uniqueValues=1000)
spec_orders = spec_orders.withColumn("customer_id", "long", minValue=1, maxValue=100)
spec_orders = spec_orders.withColumn("order_date", "timestamp", begin="2023-01-01T00:00:00", end="2024-12-31T23:59:59")
spec_orders = spec_orders.withColumn("order_amount", "decimal(10,2)", minValue=10.00, maxValue=5000.00)

df_orders = spec_orders.build()
df_orders.show(5, truncate=False)

orders_path = f"{output_location}/orders"
(df_orders.write.mode("overwrite").format(output_format).save(orders_path))

import json
import os
schema_json = df_orders.schema.json()
schema_path = f"{schema_output_location}/orders_schema.json"
try:
    dbutils.fs.put(schema_path, schema_json, overwrite=True)
except NameError:
    with open(schema_path.replace("dbfs:", ""), "w") as f:
        f.write(schema_json)

print(f"Generated orders: {df_orders.count():,} rows")

## Generate Order Details Table

In [ ]:
spec_order_details = dg.DataGenerator(spark, rows=2500, partitions=2)
spec_order_details = spec_order_details.withColumn("order_id", "long", minValue=1, maxValue=1000)
spec_order_details = spec_order_details.withColumn("product_name", "string", values=["Laptop", "Mouse", "Keyboard", "Monitor", "Headphones"], weights=[30, 20, 20, 20, 10])
spec_order_details = spec_order_details.withColumn("quantity", "int", minValue=1, maxValue=5)
spec_order_details = spec_order_details.withColumn("unit_price", "decimal(8,2)", minValue=5.00, maxValue=2000.00)

df_order_details = spec_order_details.build()
df_order_details.show(5, truncate=False)

(df_order_details.write.mode("overwrite").format(output_format).save(f"{output_location}/order_details"))
print(f"Generated order_details: {df_order_details.count():,} rows")

## Summary

In [ ]:
print("Synthetic data generation completed!")
print(f"Tables: orders, order_details")
try:
    for f in dbutils.fs.ls(output_location):
        print(f"  - {f.name}")
except NameError:
    import os
    path = output_location.replace("dbfs:", "") if output_location.startswith("dbfs:") else output_location
    if os.path.exists(path):
        for d in os.listdir(path):
            print(f"  - {d}")